# GKBA vs eGKBA — k-rep free LLG comparison

This notebook compares the `free` k-representation examples already present in the
package, using a common setup so the comparison is direct.

Physical setup:

- two-site central region, two leads
- initial local moments orthogonal: site 1 along `x`, site 2 along `z`
- smooth lead switch-on through a `sin^4` envelope
- LLG spin dynamics activated only after `t > t_llg_start`
- after LLG starts, site 1 is kept fixed along `x` and site 2 evolves freely

In [ ]:
using DifferentialEquations, LinearAlgebra
using PyPlot
using LaTeXStrings
using Tullio

import GKBA: init_gkba, init_egkba, make_llg_params,
             eom_gkba!, eom_egkba!, compute_observables!, unpack!,
             build_hs, build_hseα, heun, K_BOLTZMAN, LLGParams

In [ ]:
rc("text", usetex=true)
rc("font", family="serif")
rc("text.latex", preamble=raw"\usepackage{amsmath}")
rc("font", size=18)
rc("axes", labelsize=20, titlesize=20)
rc("xtick", labelsize=16)
rc("ytick", labelsize=16)
rc("legend", fontsize=14)

## Common parameters

The original drivers `run_gkba_krep_free.jl` and `run_egkba_krep_free.jl` use slightly
different numerical choices. Here we deliberately align the parameters and the coupling
turn-on so that the comparison reflects the approximation (`GKBA` vs `eGKBA`) rather
than driver-level differences.

In [ ]:
nx, ny   = 2, 1
ns       = nx * ny
nk       = 300
γ        = 1.0
γso      = 0.0
γc       = 1.0
j_sd     = 0.1
Temp     = 300.0
β_phys   = 1.0 / (K_BOLTZMAN * Temp)
β_egkba  = 1.0

t_0          = 0.0
dt           = 0.1
t_end        = 400.0
t_llg_start  = 200.0
ts           = collect(t_0 + dt : dt : t_end)

stepp(t; ti = 60.0) = t < ti ? sin((π / 2) * t / ti)^4 : 1.0

In [ ]:
# times = 0.0:0.1:100.0
# plt.plot(times, stepp.(times))
# plt.xlabel(L"t")
# plt.ylabel(L"g(t)")

## Helper functions

The comparison follows the `free` drivers already in the package:

- `site 1` starts and remains fixed along `x`
- `site 2` starts along `z`
- before `t_llg_start`, the moments are frozen and only the electronic problem evolves
- after `t_llg_start`, the moments evolve by LLG with electron spin torque

In [ ]:
function init_orthogonal_moments(nx, ny)
    vm = zeros(Float64, nx * ny, 3)
    vm[1, 1] = 1.0   # site 1 -> x
    vm[2, 3] = 1.0   # site 2 -> z
    vm
end

function allocate_history()
    moments = zeros(Float64, length(ts), ns, 3)
    sden    = zeros(Float64, length(ts), ns, 3)
    scurr   = zeros(Float64, length(ts), 3, 2)
    ccurr   = zeros(Float64, length(ts), 2)
    cden    = zeros(Float64, length(ts), 2 * ns)
    moments, sden, scurr, ccurr, cden
end

function make_llg_params_safe(nx, ny; dt = dt)
    # Local workaround for ny = 1: the package helper currently allocates `jys_exc`
    # with length one instead of `ny - 1 = 0`, which breaks the Kronecker sum in `heff`.
    # Keep this here until the backend helper is fixed globally.
    ε = zeros(Int, 3, 3, 3)
    ε[1, 2, 3] = ε[2, 3, 1] = ε[3, 1, 2] = 1
    ε[3, 2, 1] = ε[2, 1, 3] = ε[1, 3, 2] = -1
    ns = nx * ny
    LLGParams(
        nx = nx,
        ny = ny,
        nt = 1,
        dt = dt,
        h0_a1x = zeros(Float64, ns, 3),
        jx_exc = 0.0,
        jy_exc = 0.0,
        g_lambda = 0.0,
        j_sd = 0.0,
        j_dmi = 0.0,
        j_ani = 0.0,
        j_dem = 0.0,
        js_pol = 0.0,
        js_ana = 0.0,
        thop = 0.0,
        e_x = [0.0, 0.0, 0.0],
        e_demag_x = [0.0, 0.0, 0.0],
        js_sd_a1 = fill(0.0, ns),
        js_ani_a1 = fill(0.0, ns),
        js_dem_a1 = fill(0.0, ns),
        jxs_exc = fill(0.0, max(nx - 1, 0)),
        jys_exc = fill(0.0, max(ny - 1, 0)),
        ε = ε,
    )
end

function update_coupling!(dv, tt)
    # In k-representation the explicit tunneling amplitudes carry the switch-on.
    dv.hseα_ikα = build_hseα(nx * ny, γc, nk) .* stepp(tt)
    @tullio dv.heαs_kαi[k, α, i] := conj(dv.hseα_ikα[i, k, α])
end

function record_step!(moments, sden, scurr, ccurr, cden, n, dv, ov)
    moments[n, :, :] = real(dv.vm_i1x)
    sden[n, :, :]    = real(ov.sden_i1x)
    scurr[n, :, :]   = real(ov.scurr_xα)
    ccurr[n, :]      = real(ov.curr_α)
    cden[n, :]       = real(ov.cden_i)
end

## GKBA and eGKBA runs

In [ ]:
function run_gkba_free()
    vm0 = init_orthogonal_moments(nx, ny)
    dv, ov = init_gkba(; nx, ny, nk, γ, γso, γc, j_sd, Temp, vm_i1x = vm0)
    update_coupling!(dv, t_0)
    lv = make_llg_params_safe(nx, ny)

    prob = ODEProblem(eom_gkba!, dv.rkvec, (t_0, t_end), dv)
    integ = init(prob, Vern7(); dt = dt, save_everystep = false,
                 adaptive = true, dense = false)

    moments, sden, scurr, ccurr, cden = allocate_history()

    for (n, tt) in enumerate(ts)
        step!(integ, dt, true)
        unpack!(dv, integ.u)
        compute_observables!(ov, dv)

        if tt > t_llg_start
            dv.vm_i1x .= heun(dv.vm_i1x, ov.sden_i1x, dt, lv)
            dv.vm_i1x[1, :] .= [1.0, 0.0, 0.0]
        end

        dv.hs_ij = build_hs(dv.vm_i1x, nx, ny, γ, γso, j_sd)
        update_coupling!(dv, tt)
        record_step!(moments, sden, scurr, ccurr, cden, n, dv, ov)
    end

    moments, sden, scurr, ccurr, cden
end

function run_egkba_free()
    vm0 = init_orthogonal_moments(nx, ny)
    dv, ov = init_egkba(; nx, ny, nk, γ, γso, γc, j_sd,
                         Temp, β_override = β_egkba, vm_i1x = vm0)
    update_coupling!(dv, t_0)
    lv = make_llg_params_safe(nx, ny)

    prob = ODEProblem(eom_egkba!, dv.rkvec, (t_0, t_end), dv)
    integ = init(prob, Vern7(); dt = dt, save_everystep = false,
                 adaptive = true, dense = false)

    moments, sden, scurr, ccurr, cden = allocate_history()

    for (n, tt) in enumerate(ts)
        step!(integ, dt, true)
        unpack!(dv, integ.u)
        compute_observables!(ov, dv)

        if tt > t_llg_start
            dv.vm_i1x .= heun(dv.vm_i1x, ov.sden_i1x, dt, lv)
            dv.vm_i1x[1, :] .= [1.0, 0.0, 0.0]
        end

        dv.hs_ij = build_hs(dv.vm_i1x, nx, ny, γ, γso, j_sd)
        update_coupling!(dv, tt)
        record_step!(moments, sden, scurr, ccurr, cden, n, dv, ov)
    end

    moments, sden, scurr, ccurr, cden
end

## Run comparison

In [ ]:
println("Running GKBA free k-rep...")
@time mom_gk, sd_gk, sc_gk, cc_gk, cd_gk = run_gkba_free()

println("Running eGKBA free k-rep...")
@time mom_eg, sd_eg, sc_eg, cc_eg, cd_eg = run_egkba_free()

println("Done.")

## Local moments

In [ ]:
labels = ["x", "y", "z"]
site = 2

fig, axs = plt.subplots(1, 3, figsize = (14, 4), sharey = false)
for (ix, ax) in enumerate(axs)
    ax.plot(ts, mom_gk[:, site, ix], "b-",  lw = 2.0, label = "GKBA")
    ax.plot(ts, mom_eg[:, site, ix], "r--", lw = 1.6, label = "eGKBA")
    ax.axvline(t_llg_start, color = "0.5", ls = ":", lw = 1.0)
    ax.set_xlabel(L"t")
    ax.set_title("site $(site), component $(labels[ix])")
    ix == 1 && ax.legend(frameon = false)
end
fig.suptitle("Local moments after orthogonal initialization")
fig.tight_layout()

## Electronic spin density

In [ ]:
site = 2

fig, axs = plt.subplots(1, 3, figsize = (14, 4), sharey = false)
for (ix, ax) in enumerate(axs)
    ax.plot(ts, sd_gk[:, site, ix], "b-",  lw = 2.0, label = "GKBA")
    ax.plot(ts, sd_eg[:, site, ix], "r--", lw = 1.6, label = "eGKBA")
    ax.axvline(t_llg_start, color = "0.5", ls = ":", lw = 1.0)
    ax.set_xlabel(L"t")
    ax.set_title("site $(site), component $(labels[ix])")
    ix == 1 && ax.legend(frameon = false)
end
fig.suptitle("Electronic spin density")
fig.tight_layout()

## Spin current into lead 1

In [ ]:
lead = 1

fig, axs = plt.subplots(1, 3, figsize = (14, 4), sharey = false)
for (ix, ax) in enumerate(axs)
    ax.plot(ts, sc_gk[:, ix, lead], "b-",  lw = 2.0, label = "GKBA")
    ax.plot(ts, sc_eg[:, ix, lead], "r--", lw = 1.6, label = "eGKBA")
    ax.axvline(t_llg_start, color = "0.5", ls = ":", lw = 1.0)
    ax.set_xlabel(L"t")
    ax.set_title("lead $(lead), component $(labels[ix])")
    ix == 1 && ax.legend(frameon = false)
end
fig.suptitle("Spin current into lead 1")
fig.tight_layout()

## Charge current

In [ ]:
fig, ax = plt.subplots(figsize = (8, 4))
ax.plot(ts, cc_gk[:, 1], "b-",  lw = 2.0, label = "GKBA, lead 1")
ax.plot(ts, cc_eg[:, 1], "r--", lw = 1.6, label = "eGKBA, lead 1")
ax.plot(ts, cc_gk[:, 2], "c-",  lw = 1.6, label = "GKBA, lead 2")
ax.plot(ts, cc_eg[:, 2], "m--", lw = 1.4, label = "eGKBA, lead 2")
ax.axvline(t_llg_start, color = "0.5", ls = ":", lw = 1.0)
ax.set_xlabel(L"t")
ax.set_title("Charge current")
ax.legend(frameon = false)
fig.tight_layout()

## Notes

When reading the plots, it is useful to separate two regimes:

- before `t_llg_start`, differences come only from the electronic approximation because
  the local moments are still frozen
- after `t_llg_start`, any mismatch also feeds back into the LLG dynamics of site 2

This makes the notebook a convenient bridge between the purely electronic comparisons
and the fully coupled spin-electron dynamics used in the package drivers.